In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import ast
import json
import os
import re
import time
import subprocess
import zipfile

DATASET_DIR = '/content/drive/MyDrive/My Research/APR Research/Dataset'

KB_PATH = f'{DATASET_DIR}/pattern_knowledge_base_final.json'
TEST_PATH = f'{DATASET_DIR}/test_loop_filtered_verified.jsonl'
ZIP_PATH = f'{DATASET_DIR}/public_test_cases.zip'
EXTRACT_DIR = f'{DATASET_DIR}/test_cases'
TEST_CASES_DIR = f'{EXTRACT_DIR}/codenet/public_test_cases'

OUTPUT_PATH = f'{DATASET_DIR}/phase2_batch_results_v2.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 5

# Load Knowledge Base
with open(KB_PATH) as f:
    pattern_knowledge_base = json.load(f)
print(f'Loaded Knowledge Base: {len(pattern_knowledge_base)} patterns')

# Load test samples
test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

# Extract test cases zip if not already extracted
if not os.path.isdir(TEST_CASES_DIR):
    print('Extracting test cases zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
print(f'Test cases available for {len(os.listdir(TEST_CASES_DIR))} problems')

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN_MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"

gen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    quantization_config=gen_bnb_config,
    device_map="auto",
)


def call_gen_llm(prompt, max_new_tokens=800):
    messages = [{"role": "user", "content": prompt}]
    inputs = gen_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(gen_model.device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    input_length = inputs['input_ids'].shape[1]
    return gen_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print('DeepSeek-Coder-6.7B-Instruct loaded.')

In [ ]:
def extract_code_from_response(response):
    """
    Robustly extracts Python code from an LLM response that may have
    prose before/after a ```python ... ``` fence, or no fence at all.
    """
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []

    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases


def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)


def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(
            ['python3', '-c', code],
            input=stdin_data,
            capture_output=True,
            text=True,
            timeout=timeout_sec,
        )
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec


def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {
                **case,
                'actual_output': output if output is not None else '(crashed or timed out)',
            }
    return True, None


def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats


def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}

    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}

    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}

    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)

    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}

    speedup = slow_runtime / repaired_runtime

    if speedup >= speedup_threshold:
        return {
            'verdict': 'ACCEPT',
            'speedup': round(speedup, 3),
            'slow_runtime': round(slow_runtime, 4),
            'repaired_runtime': round(repaired_runtime, 4),
        }
    else:
        return {
            'verdict': 'RETRY',
            'reason': 'insufficient_speedup',
            'speedup': round(speedup, 3),
            'slow_runtime': round(slow_runtime, 4),
            'repaired_runtime': round(repaired_runtime, 4),
        }

print('Validation ready.')

In [ ]:
def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    completed.add(rec['problem_id'])
                except Exception:
                    continue
    return completed

In [ ]:
def build_all_patterns_context(kb):
    blocks = []
    for entry in kb:
        poi = entry.get('poi', entry)  # adjust based on Cell 9 output
        pattern_name = entry.get('pattern_name', entry.get('pattern_id', 'Unknown'))
        activation_str = "\n".join(f"  - {c}" for c in poi.get("activation_conditions", []))
        constraints_str = "\n".join(f"  - {c}" for c in poi.get("constraints", []))
        blocks.append(f"""Pattern: {pattern_name}
Transformation: {poi.get('transformation', '')}
Activation conditions:
{activation_str}
Constraints:
{constraints_str}
Expected benefit: {poi.get('expected_benefit', '')}""")
    return "\n\n".join(blocks)


ALL_PATTERNS_CONTEXT = None  # set inside run_single_llm_baseline below

def build_single_llm_prompt(slow_code):
    return f"""You are an expert Python engineer specializing in performance optimization.

Here are six known categories of loop performance optimization patterns:

{ALL_PATTERNS_CONTEXT}

Given the slow code below, first identify which pattern(s) apply, then rewrite the code
applying the appropriate transformation(s).

Slow code:
{slow_code}

Return ONLY the optimized Python code. No markdown. No explanations."""


def build_single_llm_feedback_prompt(slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')
    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {})
        feedback = (
            f"Your previous attempt failed a test case.\n"
            f"Input:\n{case.get('stdin', '')}\n"
            f"Expected output:\n{case.get('expected', '')}\n"
            f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\n"
            f"Fix the logic error."
        )
    elif reason == 'insufficient_speedup':
        feedback = (
            f"Your previous attempt was correct but NOT fast enough "
            f"({validation_result.get('speedup')}x, need >= {SPEEDUP_THRESHOLD}x). "
            f"Reconsider which pattern actually applies and try again."
        )
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_single_llm_prompt(slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback on that attempt:
{feedback}

Generate a NEW, corrected version that addresses this feedback."""


def single_llm_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    prompt = build_single_llm_prompt(slow_code)
    current_code = extract_code_from_response(call_gen_llm(prompt, max_new_tokens=800))
    history = []

    for attempt in range(1, max_attempts + 1):
        result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        result['attempt'] = attempt
        history.append(result)

        if result['verdict'] == 'ACCEPT':
            return {'final_verdict': 'ACCEPT', 'code': current_code, 'history': history}
        if attempt == max_attempts:
            return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}

        fb_prompt = build_single_llm_feedback_prompt(slow_code, current_code, result)
        current_code = extract_code_from_response(call_gen_llm(fb_prompt, max_new_tokens=800))

    return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}


def run_single_llm_baseline(test_samples, kb, test_cases_dir, output_path, max_attempts=MAX_ATTEMPTS):
    global ALL_PATTERNS_CONTEXT
    ALL_PATTERNS_CONTEXT = build_all_patterns_context(kb)

    completed = load_completed_problem_ids(output_path)
    print(f'Already completed: {len(completed)} / {len(test_samples)}')

    with open(output_path, 'a') as out_f:
        for i, sample in enumerate(test_samples):
            problem_id = sample.get('problem_id')
            slow_code = sample['input']
            if problem_id in completed:
                continue

            print(f'[{i+1}/{len(test_samples)}] {problem_id}...')
            start = time.time()
            try:
                loop_result = single_llm_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts)
                record = {
                    'problem_id': problem_id,
                    'final_verdict': loop_result['final_verdict'],
                    'num_attempts': len(loop_result['history']),
                    'history': loop_result['history'],
                    'final_code': loop_result['code'],
                    'elapsed_seconds': round(time.time() - start, 2),
                }
            except Exception as e:
                record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}

            out_f.write(json.dumps(record) + '\n')
            out_f.flush()

    print('Single-LLM baseline run complete.')

In [ ]:
print(type(pattern_knowledge_base))
if isinstance(pattern_knowledge_base, dict):
    first_key = list(pattern_knowledge_base.keys())[0]
    print("First key:", first_key)
    print(json.dumps(pattern_knowledge_base[first_key], indent=2)[:800])
else:
    print(json.dumps(pattern_knowledge_base[0], indent=2)[:800])

In [ ]:
print(list(pattern_knowledge_base["P2"].keys()))

In [ ]:
def build_all_patterns_context(kb):
    blocks = []
    for pattern_id, entry in kb.items():
        pattern_name = entry.get('pattern_name', pattern_id)
        when_str = "\n".join(f"  - {c}" for c in entry.get("when", []))
        constraints_str = "\n".join(f"  - {c}" for c in entry.get("constraints", []))
        blocks.append(f"""Pattern: {pattern_name}
Trigger conditions:
{when_str}
Rationale: {entry.get('why', '')}
Transformation: {entry.get('how', '')}
Constraints:
{constraints_str}""")
    return "\n\n".join(blocks)

In [ ]:
SINGLE_LLM_OUTPUT_PATH = f'{DATASET_DIR}/single_llm_baseline_results.jsonl'
run_single_llm_baseline(test_samples, pattern_knowledge_base, TEST_CASES_DIR, SINGLE_LLM_OUTPUT_PATH, max_attempts=MAX_ATTEMPTS)

In [ ]:
import statistics

def score_baseline_results(output_path, label):
    rows = []
    with open(output_path) as f:
        for line in f:
            rows.append(json.loads(line))

    total = len(rows)
    accepted = [r for r in rows if r.get('final_verdict') == 'ACCEPT']
    pipeline_errors = [r for r in rows if r.get('final_verdict') in ('PIPELINE_ERROR', 'PARSE_ERROR')]

    pass1_count = 0
    for r in rows:
        history = r.get('history', [])
        if not history:
            continue
        first = history[0]
        if first.get('verdict') == 'ACCEPT' or first.get('reason') == 'insufficient_speedup':
            pass1_count += 1

    print(f'=== {label} ===')
    print(f'Total: {total}')
    print(f'%OPT (accept rate): {len(accepted)/total*100:.1f}%')
    print(f'Pass@1: {pass1_count/total*100:.1f}%')
    print(f'Pipeline/parse errors: {len(pipeline_errors)}')

    if accepted:
        speedups = [r['history'][-1]['speedup'] for r in accepted]
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')
    print()
score_baseline_results(SINGLE_LLM_OUTPUT_PATH, 'Single-LLM Baseline')

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/My Research/APR Research/Dataset'
PROMPTING_OUTPUT_PATH = f'{DATASET_DIR}/prompting_baseline_results.jsonl'
SINGLE_LLM_OUTPUT_PATH = f'{DATASET_DIR}/single_llm_baseline_results.jsonl'

In [ ]:
import os
DATASET_DIR = '/content/drive/MyDrive/My Research/APR Research/Dataset'
for f in os.listdir(DATASET_DIR):
    if 'prompt' in f.lower() or 'single' in f.lower() or 'llm' in f.lower():
        print(f)

In [ ]:
def compute_k1_only(output_path, label):
    rows = []
    with open(output_path) as f:
        for line in f:
            rows.append(json.loads(line))

    total = len(rows)
    opt_k1 = 0
    pass1_k1 = 0

    for r in rows:
        history = r.get('history', [])
        if not history:
            continue
        first = history[0]
        if first.get('verdict') == 'ACCEPT':
            opt_k1 += 1
        if first.get('verdict') == 'ACCEPT' or first.get('reason') == 'insufficient_speedup':
            pass1_k1 += 1

    print(f'=== {label} (k=1, first attempt only) ===')
    print(f'%OPT@1: {100*opt_k1/total:.1f}%')
    print(f'Pass@1: {100*pass1_k1/total:.1f}%')
    print()

compute_k1_only(PROMPTING_OUTPUT_PATH, 'Prompting')
compute_k1_only(SINGLE_LLM_OUTPUT_PATH, 'Single-LLM')

In [ ]:
import json, statistics

def k1_speedup(path):
    results = []
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))

    speedups = []
    pass1_k1 = 0
    for r in results:
        history = r.get('history', [])
        if not history:
            continue
        first = history[0]
        if first.get('verdict') == 'ACCEPT':
            speedups.append(first['speedup'])
        if first.get('verdict') == 'ACCEPT' or first.get('reason') == 'insufficient_speedup':
            pass1_k1 += 1

    if speedups:
        print(f'{path}')
        print(f'  n (accepted@k=1) = {len(speedups)}')
        print(f'  Median speedup: {statistics.median(speedups):.3f}x')
        print(f'  Mean speedup: {statistics.mean(speedups):.3f}x')
        print(f'Pass@1: {100*pass1_k1/total:.1f}%')
    else:
        print(f'{path}: no k=1 accepts found')
    print()

# Ours
k1_speedup(OUTPUT_PATH)  # ba tomar pipeline-er result file-er path

# Prompting
k1_speedup(PROMPTING_OUTPUT_PATH)

# Single-LLM
k1_speedup(SINGLE_LLM_OUTPUT_PATH)

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/My Research/APR Research/Dataset'
OUTPUT_PATH_OURS = f'{DATASET_DIR}/phase2_batch_results_test.jsonl'

In [ ]:
import statistics

def k1_speedup(path, label):
    results = []
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))

    speedups = []
    for r in results:
        history = r.get('history', [])
        if not history:
            continue
        first = history[0]
        if first.get('verdict') == 'ACCEPT':
            speedups.append(first['speedup'])

    print(f'=== {label} (k=1) ===')
    print(f'n (accepted@k=1): {len(speedups)}')
    if speedups:
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')
    else:
        print('No accepted samples at k=1')
    print()

k1_speedup(OUTPUT_PATH_OURS, 'Ours')

In [ ]:
results = []
with open(OUTPUT_PATH_OURS) as f:
    for line in f:
        results.append(json.loads(line))

total = len(results)
pass1_count = 0

for r in results:
    history = r.get('history', [])
    if not history:
        continue
    first = history[0]  # first attempt = the "k=1" attempt
    if first.get('verdict') == 'ACCEPT' or first.get('reason') == 'insufficient_speedup':
        pass1_count += 1

print(f'Total: {total}')
print(f'Pass@1 count (first attempt functionally correct): {pass1_count}')
print(f'Pass@1 (k=1): {100*pass1_count/total:.2f}%')

In [ ]:
import statistics

def full_k7_metrics(path, label):
    results = []
    with open(path) as f:
        for line in f:
            results.append(json.loads(line))

    total = len(results)
    accepted = [r for r in results if r.get('final_verdict') == 'ACCEPT']
    pass1_count = sum(
        1 for r in results
        if r.get('history') and (
            r['history'][0].get('verdict') == 'ACCEPT' or
            r['history'][0].get('reason') == 'insufficient_speedup'
        )
    )

    print(f'=== {label} ===')
    print(f'Total: {total}')
    print(f'%OPT: {100*len(accepted)/total:.1f}%')
    print(f'Pass@1: {100*pass1_count/total:.1f}%')

    speedups = [r['history'][-1]['speedup'] for r in accepted if 'speedup' in r['history'][-1]]
    if speedups:
        print(f'Median speedup: {statistics.median(speedups):.3f}x')
        print(f'Mean speedup: {statistics.mean(speedups):.3f}x')
    print()

# path-ta tomar k=7-run-er actual file diye replace koro
K7_OUTPUT_PATH = f'{DATASET_DIR}/phase2_batch_results_test.jsonl'  # ba je file-e k=7 result ache
full_k7_metrics(K7_OUTPUT_PATH, 'Ours (k=7)')